# Explore the pipeline
All the code lives in the `analog_engine` package; this notebook only runs steps and looks at results.
From a terminal the same thing is `python -m analog_engine [download planets base score frontend]`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "analog_engine").is_dir():  # works from AnalogEngine/, notebooks/ or scripts/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from analog_engine import filters, likeness, pipeline, plots, scoring, validation
from analog_engine.config import PROCESSED, RAW

## Run steps (comment out what is already up to date)

In [ ]:
pipeline.download()                  # raw data, skips files on disk
pipeline.run_planets()               # Moon / Mars terrain tables
pipeline.run_base()                  # Earth table + filters + likeness -> cells.parquet
pipeline.run_score()                 # applications -> cells_scored.parquet
pipeline.run_frontend()              # frontend_cells.json

↓ downloading wc2.1_10m_bio.zip ...
✓ unzipped WorldClim


/home/mmacura/miniforge3/envs/spaceapps/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Enter your Earthdata Login username:  nasa.routing763@passmail.net
Enter your Earthdata password:  ········


QUEUEING TASKS | : 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 753.83it/s]
PROCESSING TASKS | : 100%|███████████████████████████████████████████████████████████████| 1/1 [00:16<00:00, 16.84s/it]
COLLECTING RESULTS | : 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 10837.99it/s]


✓ downloaded MCD12C1.A2022001.061.2023244164746.hdf
picked: GEDI_HighQuality_Shots_Rasters.gediv002_agbd-a0-qf_vf_20190417_20230316_12000m.tif


QUEUEING TASKS | : 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 485.56it/s]
PROCESSING TASKS | : 100%|███████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.36s/it]
COLLECTING RESULTS | : 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 12018.06it/s]


↓ downloading ETOPO_2022_v1_15s_N00E000_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00E015_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00E030_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00E045_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00E060_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00E075_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00E090_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00E105_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00E120_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00E135_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00E150_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00E165_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00W015_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00W030_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00W045_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00W060_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00W075_surface.tif ...
↓ downloading ETOPO_2022_v1_15s_N00W090_surface.

## Screening

In [ ]:
cells = pd.read_parquet(PROCESSED / "cells.parquet")
display(filters.funnel(cells))
display(validation.check_screening(cells))
plots.screening_map(cells, validation.screening_sites()); plt.show()

## Moon / Mars likeness

In [ ]:
matches = pd.read_parquet(PROCESSED / "site_matches.parquet")
display(matches[matches["rank"] == 1].drop(columns=["rank", "site_cell_id"]
        + [c for c in matches if c.startswith("dz_")]).round(2))
for body in likeness.BODIES:
    print(f"Most {body}-like Earth cells:")
    display(cells.nlargest(5, f"{body}_like")[["cell_id", "lat", "lon", *likeness.TOPO_FEATURES, f"{body}_like"]].round(2))

## Applications

In [ ]:
scored = pd.read_parquet(PROCESSED / "cells_scored.parquet")
apps = [a.name for a in scoring.APPLICATIONS]
for app in scoring.APPLICATIONS:
    print(app.name)
    display(scoring.top(scored, app))
display(validation.check_applications(scored, apps))

In [ ]:
plots.label_maps(scored, apps, validation.application_sites()); plt.show()
plots.quality_maps(scored, apps); plt.show()